## 다양한 통계적 기법을 활용한 EDA


## 주요 가설 수립을 위한 가설 후보군 탐색


## 자동화


In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu, ks_2samp, f_oneway
from sklearn.preprocessing import StandardScaler

from utils import *

pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_columns', None)

In [4]:
df = pd.read_csv('data/fraud_oracle.csv')

In [5]:
time_vars = ["Month", "WeekOfMonth", "DayOfWeek", "DayOfWeekClaimed", 'MonthClaimed', 'WeekOfMonthClaimed']
vehicle_vars = ["Make", "VehiclePrice", "VehicleCategory", "AgeOfVehicle"]
personal_vars = ["Sex", 'MaritalStatus', "Age", 'DriverRating', 'AgeOfPolicyHolder', 'NumberOfCars', 'PastNumberOfClaims']
policy_vars = ["PolicyType", 'Deductible', 'AgentType', "NumberOfSuppliments", "BasePolicy"]
accident_vars = ['Days_Policy_Accident', 'Days_Policy_Claim', 'PoliceReportFiled', 'WitnessPresent', 'AddressChange_Claim']

# 어떻게 검증하지?

### Numerical vs Numerical
- t-test
- Kolmogorov-Smirnov Test
- Correlation

### Numerical vs Categorical (3개 이상)
- ANOVA

### Categorical vs Categorical
- Chi-square

In [6]:
from utils import plot_dataframe

In [7]:
plot_dataframe(df[vehicle_vars+['FraudFound_P']])

/Users/fastriver/Desktop/메타코드 SQL파이썬 실습/웹로그 데이터분석/web_log_analysis_env/lib/python3.13/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [8]:
def chi_square(array1, array2):
    """Categorical vs Categorical"""
    contingency_table = pd.crosstab(array1, array2)
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    return chi2, p, dof, expected, contingency_table

In [9]:
chi2, p, dof, expected, contingency_table = chi_square(df['FraudFound_P'], df['AgeOfVehicle'])

In [10]:
p

np.float64(0.0026129967669177635)

In [11]:
contingency_table

AgeOfVehicle,2 years,3 years,4 years,5 years,6 years,7 years,more than 7,new
FraudFound_P,,,,,,,,
0,70,139,208,1262,3220,5482,3775,341
1,3,13,21,95,228,325,206,32


In [12]:
expected

array([[6.86304150e+01, 1.42901686e+02, 2.15292672e+02, 1.27577361e+03,
        3.24161193e+03, 5.45940850e+03, 3.74270798e+03, 3.50673217e+02],
       [4.36958495e+00, 9.09831388e+00, 1.37073281e+01, 8.12263943e+01,
        2.06388067e+02, 3.47591505e+02, 2.38292023e+02, 2.23267834e+01]])

In [13]:
def numerical_test(array1, array2):
    """
    Numerical vs Numerical
    """
    scaler = StandardScaler()
    array = np.concatenate([array1, array2])
    array = scaler.fit_transform(array.reshape(-1, 1)).flatten()

    array1_norm = array[:len(array1)]
    array2_norm = array[len(array1):]

    t_stat, t_p = ttest_ind(array1_norm, array2_norm)
    ks_stat, ks_p = ks_2samp(array1_norm, array2_norm)

    return t_stat, t_p, ks_stat, ks_p

In [14]:
relevant_vars = list()

for var in personal_vars:
    print(var)
    if pd.api.types.is_numeric_dtype(df[var]):
        non_fraud = df.loc[df['FraudFound_P']==0, var].values
        fraud = df.loc[df['FraudFound_P']==1, var].values

        t_stat, t_p, ks_stat, ks_p = numerical_test(non_fraud, fraud)
        
        if (t_p<0.05) and (ks_p<0.05):
            relevant_vars.append(var)
            print(t_p, ks_p)
    elif pd.api.types.is_object_dtype(df[var]):
        chi2, p, dof, expected, contingency_table = chi_square(df['FraudFound_P'], df[var])
        if (p<0.05):
            relevant_vars.append(var)
            print(p)

    print('-'*30)


Sex
0.00023985178051231303
------------------------------
MaritalStatus
------------------------------
Age
0.00022102056801804888 0.0014187232697535769
------------------------------
DriverRating
------------------------------
AgeOfPolicyHolder
6.150519521424798e-05
------------------------------
NumberOfCars
------------------------------
PastNumberOfClaims
1.4337179439711134e-11
------------------------------


In [15]:
relevant_vars

['Sex', 'Age', 'AgeOfPolicyHolder', 'PastNumberOfClaims']

---

In [16]:
from utils2 import plot_dataframe

In [17]:
plot_dataframe(df[relevant_vars + ["FraudFound_P"]], color_column="FraudFound_P")

FraudFound_P


FraudFound_P


FraudFound_P
